## Stage 0 — Setup

Imports and header so worth athletics doesn't pick it up as a bot.

In [35]:
import requests
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
from io import StringIO

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

## Stage 1 — Scrape a single race
Here we scrap the races from the results page and get this information:
athlete, country, time, record flag 
- reaction time
- wind reading
- date, venue, meet name

Handles non-numeric results (DNF/DQ/DNS) by keeping the row with a NaN time,
since a DNF is still a real data point (something went wrong that race).

In [36]:
def clean_mark(mark):
    """Split '9.79 PB' into (9.79, 'PB'), or '9.91' into (9.91, None).
    Non-numeric marks (DNF/DQ/DNS) get NaN time and the code as the flag."""
    if pd.isna(mark):
        return np.nan, None
    parts = str(mark).strip().split() # whitespace
    try:
        time = float(parts[0]) 
        flag = parts[1] if len(parts) > 1 else None
    except ValueError:
        time = np.nan
        flag = parts[0]  # e.g. 'DNF', 'DQ', 'DNS'
    return time, flag


def scrape_race(url):
    resp = requests.get(url, headers=HEADERS, timeout=15) # timeout in seconds
    resp.raise_for_status() # raises an error if the request failed

    tables = pd.read_html(StringIO(resp.text)) # parse all tables from the HTML
    results_table = max(tables, key=len) # get largest tble
    
    # Normalize column names across inconsistent page formats
    results_table.columns = [str(c).strip() for c in results_table.columns]
    rename_map = {
        "Mark": "MARK",
        "Athlete": "ATHLETE",
        "Unnamed: 2": "COUNTRY",
        }
    results_table = results_table.rename(columns=rename_map)

    results_table[["time", "record_flag"]] = results_table["MARK"].apply(
        lambda m: pd.Series(clean_mark(m))
    )

    soup = BeautifulSoup(resp.text, "html.parser") # parse the HTML with BeautifulSoup

    # wind
    heading = soup.find(string=lambda s: s and "Wind" in s) # find wind 
    wind = None
    if heading:
        match = re.search(r"Wind\s*([+-]?\d+\.\d+)", heading) # regex for finding wind speed
        if match:
            wind = float(match.group(1))

    # date — from EventDate meta tag 
    date = None
    for meta in soup.find_all("meta"):
        if meta.get("name") == "EventDate" or meta.get("property") == "EventDate":
            date = meta.get("content")
            break

    # venue + meet name
    desc_tag = soup.find("meta", attrs={"property": "og:description"})
    venue = None
    if desc_tag and desc_tag.get("content"):
        venue_match = re.search(r"in\s+([A-Za-z\s]+)$", desc_tag["content"].strip())
        if venue_match:
            venue = venue_match.group(1).strip()

    title_tag = soup.find("meta", attrs={"property": "og:title"})
    meet_name = title_tag["content"].strip() if title_tag and title_tag.get("content") else None

    results_table["wind"] = wind
    results_table["date"] = date
    results_table["venue"] = venue
    results_table["meet_name"] = meet_name
    results_table["source_url"] = url

    return results_table[["ATHLETE", "COUNTRY", "time", "record_flag", "Reaction Time",
                            "wind", "date", "venue", "meet_name", "source_url"]]

## Stage 2 — Test across multiple races

One race working isn't enough proof — different meets can have messier data
(DNF/DQ results) or slightly different page structures. Here we run the
scraper across 4 different races to find where it breaks.

In [37]:
urls = [
    "https://worldathletics.org/results/olympic-games/2024/the-xxxiii-olympic-games-7153115/men/100-metres/final/result",  # Paris 2024
    "https://worldathletics.org/results/world-athletics-championships/2023/world-athletics-championships-budapest-2023-7138987/men/100-metres/final/result",  # Budapest 2023
    "https://worldathletics.org/results/olympic-games/2021/the-xxxii-olympic-games-7132391/men/100-metres/final/result",  # Tokyo 2020(21) - has a DNF
    "https://worldathletics.org/competitions/world-athletics-championships/world-athletics-championships-oregon-2022-7137279/results/men/100-metres/final/result",  # Oregon 2022 - different URL structure
]

all_races = []
for u in urls:
    try:
        race = scrape_race(u)
        all_races.append(race)
        print(f"OK ({len(race)} rows): {u}")
    except Exception as e:
        print(f"FAILED: {u}")
        print(f"   error: {e}")

full_dataset = pd.concat(all_races, ignore_index=True)
full_dataset

OK (8 rows): https://worldathletics.org/results/olympic-games/2024/the-xxxiii-olympic-games-7153115/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/results/world-athletics-championships/2023/world-athletics-championships-budapest-2023-7138987/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/results/olympic-games/2021/the-xxxii-olympic-games-7132391/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/competitions/world-athletics-championships/world-athletics-championships-oregon-2022-7137279/results/men/100-metres/final/result


C:\Users\rotem\AppData\Local\Temp\ipykernel_17640\656437860.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full_dataset = pd.concat(all_races, ignore_index=True)


,ATHLETE,COUNTRY,time,record_flag,Reaction Time,wind,date,venue,meet_name,source_url
0,Noah LYLES,USA,9.79,PB,0.178,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
1,Kishane THOMPSON,JAM,9.79,NaN,0.176,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
2,Fred KERLEY,USA,9.81,SB,0.108,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
3,Akani SIMBINE,RSA,9.82,NR,0.149,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
4,Lamont Marcell JACOBS,ITA,9.85,SB,0.114,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
5,Letsile TEBOGO,BOT,9.86,NR,0.178,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
6,Kenneth BEDNAREK,USA,9.88,NaN,0.163,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
7,Oblique SEVILLE,JAM,9.91,NaN,0.171,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
8,Noah LYLES,USA,9.83,=WL,0.145,0.0,19/08/2023 00:00:00,Budapest,World Athletics | 100 Metres Result | World A...,https://worldathletics.org/results/world-athle...
9,Letsile TEBOGO,BOT,9.88,NR,0.129,0.0,19/08/2023 00:00:00,Budapest,World Athletics | 100 Metres Result | World A...,https://worldathletics.org/results/world-athle...


## Stage 3 — Handle inconsistent page formats

Running the scraper across multiple races (Stage 2) revealed that not every
World Athletics results page is structured the same way. Two real bugs
turned up:

1. Tokyo 2020(21) had a DNF result — 'MARK' contained the text "DNF" instead
   of a time, which crashed the original clean_mark() function. Fixed by
   catching non-numeric marks and storing them as a record_flag instead of
   a time.

2. Oregon 2022 uses a different URL structure (/competitions/.../results/...
   instead of /results/...), and its results table uses different column
   names ('Mark' instead of 'MARK',

In [38]:
resp = requests.get(
    "https://worldathletics.org/competitions/world-athletics-championships/world-athletics-championships-oregon-2022-7137279/results/men/100-metres/final/result",
    headers=HEADERS, timeout=15
)
tables = pd.read_html(StringIO(resp.text))
print(f"Found {len(tables)} tables")

results_table = max(tables, key=len)
print(results_table.columns.tolist())
results_table

Found 1 tables
['Pos', 'Bib', 'Unnamed: 2', 'Athlete', 'Mark', 'Reaction Time']


,Pos,Bib,Unnamed: 2,Athlete,Mark,Reaction Time
0,1,2389,USA,Fred KERLEY,9.86,0.119
1,2,2360,USA,Marvin BRACY,9.88,0.118
2,3,2363,USA,Trayvon BROMELL,9.88,0.110
3,4,2039,JAM,Oblique SEVILLE,9.97,0.154
4,5,2266,RSA,Akani SIMBINE,10.01,0.129
5,6,2368,USA,Christian COLEMAN,10.01,0.104
6,7,2068,JPN,Abdul Hakim SANI BROWN,10.06,0.147
7,8,1639,CAN,Aaron BROWN,10.07,0.155


## Stage 4 — Build the real dataset

Collect a batch of race URLs (Olympic finals, World Champs finals, a few
Diamond League races), run scrape_race() across all of them, and save
the combined result to a CSV so we're not re-scraping every time we
restart the kernel.

Known gap: pages using the `/competitions/.../results/...` URL structure
(e.g. Oregon 2022) don't return wind/date/venue — only athlete/time/reaction
time. Accepted as a limitation for now rather than fixed.

In [39]:
urls = [
    "https://worldathletics.org/results/olympic-games/2024/the-xxxiii-olympic-games-7153115/men/100-metres/final/result",  # Paris 2024
    "https://worldathletics.org/results/world-athletics-championships/2023/world-athletics-championships-budapest-2023-7138987/men/100-metres/final/result",  # Budapest 2023
    "https://worldathletics.org/results/olympic-games/2021/the-xxxii-olympic-games-7132391/men/100-metres/final/result",  # Tokyo 2020(21)
    "https://worldathletics.org/competitions/world-athletics-championships/world-athletics-championships-oregon-2022-7137279/results/men/100-metres/final/result",  # Oregon 2022 - known gap: no wind/date/venue
    "https://worldathletics.org/results/world-athletics-championships/2019/iaaf-world-athletics-championships-doha-2019-7125365/men/100-metres/final/result",  # Doha 2019
    "https://worldathletics.org/results/world-athletics-championships/2017/iaaf-world-championships-london-2017-7093740/men/100-metres/final/result", 
    "https://worldathletics.org/results/iaaf-world-championships-in-athletics/2015/15th-iaaf-world-championships-7078726/men/100-metres/final/result",  # Beijing 2015 - Bolt beats Gatlin by 0.01s
    "https://worldathletics.org/results/world-athletics-championships/2013/14th-iaaf-world-championships-7003368/men/100-metres/final/result",  # Moscow 2013
    "https://worldathletics.org/results/olympic-games/2012/the-xxx-olympic-games-6999193/men/100-metres/final/result",  # London 2012 - Bolt's 9.63 Olympic Record
    "https://worldathletics.org/results/world-athletics-championships/2011/13th-iaaf-world-championships-in-athletics-7003367/men/100-metres/final/result",  # Daegu 2011 - Bolt DQ'd for false start# London 2017 - Bolt's last individual race
]

all_races = []
for u in urls:
    try:
        race = scrape_race(u)
        all_races.append(race)
        print(f"OK ({len(race)} rows): {u}")
    except Exception as e:
        print(f"FAILED: {u}")
        print(f"   error: {e}")

full_dataset = pd.concat(all_races, ignore_index=True)
full_dataset

OK (8 rows): https://worldathletics.org/results/olympic-games/2024/the-xxxiii-olympic-games-7153115/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/results/world-athletics-championships/2023/world-athletics-championships-budapest-2023-7138987/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/results/olympic-games/2021/the-xxxii-olympic-games-7132391/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/competitions/world-athletics-championships/world-athletics-championships-oregon-2022-7137279/results/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/results/world-athletics-championships/2019/iaaf-world-athletics-championships-doha-2019-7125365/men/100-metres/final/result
OK (8 rows): https://worldathletics.org/results/world-athletics-championships/2017/iaaf-world-championships-london-2017-7093740/men/100-metres/final/result
OK (9 rows): https://worldathletics.org/results/iaaf-world-championships-in-athletics/2015/15

C:\Users\rotem\AppData\Local\Temp\ipykernel_17640\2088878736.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  full_dataset = pd.concat(all_races, ignore_index=True)


,ATHLETE,COUNTRY,time,record_flag,Reaction Time,wind,date,venue,meet_name,source_url
0,Noah LYLES,USA,9.79,PB,0.178,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
1,Kishane THOMPSON,JAM,9.79,NaN,0.176,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
2,Fred KERLEY,USA,9.81,SB,0.108,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
3,Akani SIMBINE,RSA,9.82,NR,0.149,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
4,Lamont Marcell JACOBS,ITA,9.85,SB,0.114,1.0,01/08/2024 00:00:00,Paris,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
...,...,...,...,...,...,...,...,...,...,...
76,Christophe LEMAITRE,FRA,10.19,NaN,0.179,-1.4,27/08/2011 00:00:00,None,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
77,Daniel Bailey,ANT,10.26,NaN,0.165,-1.4,27/08/2011 00:00:00,None,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
78,Jimmy VICAUT,FRA,10.27,NaN,0.162,-1.4,27/08/2011 00:00:00,None,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
79,Nesta CARTER,JAM,10.95,NaN,0.154,-1.4,27/08/2011 00:00:00,None,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...


## Stage 5 — Feature engineering

Load the saved dataset and start turning race-level rows into athlete-level
features. First step: clean up data types (dates, names) so grouping by
athlete actually works correctly — then build the simplest useful feature
(recency-weighted average time) before anything fancier.

In [40]:
df = pd.read_csv("100m_races_dataset.csv")

# Normalize athlete names so the same person isn't split into two
# groups due to inconsistent capitalization (e.g. "Noah LYLES" vs "Noah Lyles")
df["ATHLETE"] = df["ATHLETE"].str.strip().str.title()

# Convert date column from text to an actual datetime, so we can sort
# chronologically and compute "most recent races"
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y %H:%M:%S", errors="coerce")

print(df["ATHLETE"].nunique(), "unique athletes")
print(df["date"].min(), "to", df["date"].max())
df.sort_values("date").head(10)

40 unique athletes
2011-08-27 00:00:00 to 2024-08-01 00:00:00


,ATHLETE,COUNTRY,time,record_flag,Reaction Time,wind,date,venue,meet_name,source_url
80,Usain Bolt,JAM,NaN,DQ,-0.104,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
73,Yohan Blake,JAM,9.92,SB,0.174,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
74,Walter Dix,USA,10.08,NaN,0.175,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
75,Kim Collins,SKN,10.09,NaN,0.155,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
79,Nesta Carter,JAM,10.95,NaN,0.154,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
77,Daniel Bailey,ANT,10.26,NaN,0.165,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
78,Jimmy Vicaut,FRA,10.27,NaN,0.162,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
76,Christophe Lemaitre,FRA,10.19,NaN,0.179,-1.4,2011-08-27,NaN,World Athletics | 100 Metres Result | 13th IA...,https://worldathletics.org/results/world-athle...
72,Tyson Gay,USA,NaN,DQ,0.145,1.5,2012-07-27,NaN,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...
71,Asafa Powell,JAM,11.99,NaN,0.155,1.5,2012-07-27,NaN,World Athletics | 100 Metres Result | The XXX...,https://worldathletics.org/results/olympic-gam...


In [41]:
race_counts = df.groupby("ATHLETE").size().sort_values(ascending=False)
race_counts.head(15)

ATHLETE
Akani Simbine             5
Justin Gatlin             5
Usain Bolt                5
Christian Coleman         4
Yohan Blake               4
Jimmy Vicaut              3
Bingtian Su               3
Andre De Grasse           3
Zharnel Hughes            3
Oblique Seville           3
Fred Kerley               3
Aaron Brown               2
Christophe Lemaitre       2
Asafa Powell              2
Abdul Hakim Sani Brown    2
dtype: int64

### Feature 1 — Per-athlete average time and consistency

For athletes with multiple races, compute their average time and the
standard deviation of their times (how consistent they are). Athletes
with only 1 race get a mean but no meaningful std — flagged separately
rather than treated as "perfectly consistent."

In [42]:
athlete_stats = df.groupby("ATHLETE")["time"].agg(
    races="count",
    avg_time="mean",
    consistency="std"
).sort_values("avg_time")

athlete_stats

,races,avg_time,consistency
ATHLETE,,,
Usain Bolt,4,9.785000,0.131022
Kishane Thompson,1,9.790000,NaN
Noah Lyles,2,9.810000,0.028284
Lamont Marcell Jacobs,2,9.825000,0.035355
Fred Kerley,3,9.836667,0.025166
Justin Gatlin,5,9.850000,0.056125
Letsile Tebogo,2,9.870000,0.014142
Kenneth Bednarek,1,9.880000,NaN
Ryan Bailey,1,9.880000,NaN
